<a href="https://colab.research.google.com/github/Existanze54/sirius-neural-networks-2026/blob/main/Homeworks/HW6_RNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Домашка 6. RNN

### Импорты и вот это всё

In [ ]:
import copy
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.optim import Adam
from torch.nn import MSELoss
from torch.nn import Module

from tqdm.auto import trange

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Классификация коротких нуклеотидов на данных PBM

**Protein Binding Microarray (PBM)** — один из методов изучения связывания транскрипционных факторов с ДНК. Результаты таких экспериментов позволяют устанавливать мотивы связывания, вроде этого:  
<img src="https://mex.autosome.org/motifs/motif401371.svg" width='300'>

Обычно мотивы объясняют экспериментальные результаты довольно хорошо, но порой алгоритмы ML могут найти более сложные зависимости. В данной задаче предлагается классифицировать олигонуклеотиды на связывающиеся и несвязывающиеся (согласно результатам PBM) с транскрипционным фактором [GCM1](https://www.uniprot.org/uniprotkb/Q9NP62/entry) человека.

In [ ]:
%%bash
FILEID=1tmbIBuPXhPMjcvAgfs6DFWakOjH6We7O
wget -q --no-check-certificate "https://docs.google.com/uc?export=download&id=$FILEID" -O gcm1.tsv

### Посмотрим на данные

In [ ]:
df = pd.read_csv('gcm1.tsv', sep='\t')
df.head(2)

`pbm_sequence` это тот самый изучаемый олигонуклеотид. `linker_sequence` — последовательность, которой нуклеотид крепится к микрочипу. `mean_signal_intensity` — нормализованная сила флуоресценции, она таки будет нашей целевой переменной.

## Задача 1. Подготовка данных

Разобьем данные сразу, чтобы избежать утечки данных в ходе аугментации.

In [ ]:
df = df.sample(frac=1.0, random_state=0)

i1, i2 = round(len(df)*0.7), round(len(df)*0.7) + round(len(df)*0.15)

train = df.iloc[:i1].copy()
val = df.iloc[i1:i2].copy()
test = df.iloc[i2:].copy()

Теперь необходимо подготовить признаки. Надо присоединить линкер к изучаемой последовательности с 5'-конца, аугментировать выборки обратно-комплементарными последовательностями, `one-hot`-закодировать все полученные олигонуклеотиды. Для этого дополните функции ниже и примените к частям датасета.

In [ ]:
def merge_linker(probe, linker):
    # your code here
    return

def get_complement(seq):
    # your code here
    return

def onehot_encode(seq):
    # your code here
    return

def prep_features(df):
    # your code here
    return

In [ ]:
X_train = prep_features(train)
X_val = prep_features(val)
X_test = prep_features(test)

Извлеките сигнал PBM и не забудьте продублировать для добавленных последовательностей.

In [ ]:
# your code here

Конвертируйте данные в тензоры и по необходимости измените их размерность и переместите на GPU.

In [ ]:
# your code here

In [ ]:
# @title Класс TrainableNN
# @markdown Здесь для вас написан класс, упрощающий обучение нейросети.
# @markdown Его можно использовать с любой архитектурой.
# @markdown Модифицируйте по необходимости.

class TrainableNN(Module):
    def compile(self, optimizer_class, criterion_class,
                lr=0.001, device=None, device_id=0,
                **optim_kwargs):
        if device is None:
            if torch.cuda.is_available():
                self.device = torch.device(f'cuda:{device_id}')
            else:
                self.device = torch.device('cpu')
        else:
            self.device = torch.device(device)
        self.to(self.device)
        self.optimizer = optimizer_class(self.parameters(), lr=lr, **optim_kwargs)
        self.criterion = criterion_class()

    def _move_to_device(self, x, y=None):
        x = x.to(self.device, non_blocking=True)
        if y is None:
            return x
        y = y.to(self.device, non_blocking=True)
        return x, y

    def _iter_batches(self, x, y=None, batch_size=None, shuffle=False):
        n = x.shape[0]
        if batch_size is None or batch_size >= n:
            if y is None:
                yield x
            else:
                yield x, y
            return

        if shuffle:
            idx = torch.randperm(n, device=x.device)
        else:
            idx = torch.arange(n, device=x.device)

        for i in range(0, n, batch_size):
            batch_idx = idx[i:i + batch_size]
            if y is None:
                yield x[batch_idx]
            else:
                yield x[batch_idx], y[batch_idx]

    def _train_step(self, x, y, batch_size=None, shuffle=True):
        self.train()
        total_loss = 0.0
        n = x.shape[0]

        for xb, yb in self._iter_batches(x, y, batch_size=batch_size, shuffle=shuffle):
            self.optimizer.zero_grad()
            pred = self(xb)
            loss = self.criterion(pred, yb)
            loss.backward()
            self.optimizer.step()
            total_loss += loss.item() * xb.shape[0]

        return total_loss / n

    def _val_step(self, x, y, batch_size=None):
        self.eval()
        total_loss = 0.0
        n = x.shape[0]

        with torch.no_grad():
            for xb, yb in self._iter_batches(x, y, batch_size=batch_size, shuffle=False):
                pred = self(xb)
                loss = self.criterion(pred, yb)
                total_loss += loss.item() * xb.shape[0]

        return total_loss / n

    def fit(self, x_train, y_train, x_val=None, y_val=None,
            num_epochs=100, batch_size=None, patience=10,
            shuffle=True):
        x_train, y_train = self._move_to_device(x_train, y_train)
        if x_val is not None and y_val is not None:
            x_val, y_val = self._move_to_device(x_val, y_val)

        self.train_losses = []
        self.val_losses = []
        best_loss = float('inf')
        best_state = copy.deepcopy(self.state_dict())
        plateau = 0

        for _ in trange(num_epochs):
            train_loss = self._train_step(x_train, y_train, batch_size=batch_size, shuffle=shuffle)
            self.train_losses.append(train_loss)

            if x_val is not None and y_val is not None:
                val_loss = self._val_step(x_val, y_val, batch_size=batch_size)
                monitor_loss = val_loss
            else:
                val_loss = None
                monitor_loss = train_loss

            self.val_losses.append(val_loss)

            if monitor_loss < best_loss:
                best_loss = monitor_loss
                best_state = copy.deepcopy(self.state_dict())
                plateau = 0
            else:
                plateau += 1

            if plateau >= patience:
                break

        self.load_state_dict(best_state)
        return {'train_loss': self.train_losses, 'val_loss': self.val_losses}

    def predict(self, x, batch_size=None):
        x = self._move_to_device(x)
        self.eval()
        preds = []

        with torch.no_grad():
            for xb in self._iter_batches(x, batch_size=batch_size, shuffle=False):
                preds.append(self(xb))

        return torch.cat(preds, dim=0)

## Задача 2. CNN

Здесь приведен пример архитектуры CNN (с более-менее оптимальными гиперпараметрами) и ее обучения на данных PBM.
Внимательно ознакомьтесь с ее блоками и интерфейсом применения `TrainableNN`.

In [ ]:
class STEMBlock(nn.Module):
    """
    Начальный блок, извлекающий простейшие k-меры в качестве эмбеддинга.
    """
    def __init__(self, in_channels, out_channels):
        super().__init__()
        c = out_channels // 4
        self.c3 = nn.Conv1d(in_channels, c, 3, padding=1, bias=False)
        self.c5 = nn.Conv1d(in_channels, c, 5, padding=2, bias=False)
        self.c7 = nn.Conv1d(in_channels, c, 7, padding=3, bias=False)
        self.c11 = nn.Conv1d(in_channels, c, 11, padding=5, bias=False)
    def forward(self, x):
        return torch.cat([self.c3(x), self.c5(x), self.c7(x), self.c11(x)], dim=1)

class ResConvBlock(nn.Module):
    """
    Основной блок, который можно повторять желаемое число раз.
    """
    def __init__(self, channels, kernel_size=3):
        super().__init__()
        p = kernel_size // 2
        self.conv = nn.Conv1d(channels, channels, kernel_size, padding=p)
        self.act = nn.ReLU()
    def forward(self, x):
        return self.act(self.conv(x) + x)

class MLPBlock(nn.Module):
    """
    Полносвязный блок для финального предсказания.
    """
    def __init__(self, in_dim, hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )
    def forward(self, x):
        return self.net(x)

class CNNRegressor(TrainableNN):
    def __init__(self, input_dim, seq_len,
                 mlp_hidden=128, channels=128,
                 num_res_blocks=3, kernel_size=7):
        super().__init__()
        self.stem = STEMBlock(input_dim, channels)
        self.res = nn.Sequential(*[
            ResConvBlock(channels, kernel_size) for _ in range(num_res_blocks)
        ])
        self.pool = nn.MaxPool1d(seq_len)
        self.mlp = MLPBlock(channels, mlp_hidden)
    def forward(self, x):
        x = self.stem(x)
        x = self.res(x)
        x = self.pool(x).squeeze(-1)
        return self.mlp(x)

In [ ]:
channels = 64
mlp_hidden = 256
num_res_blocks = 4
kernel_size = 11
batch_size = 512
lr = 1e-3

patience=20
num_epochs=200

model = CNNRegressor(input_dim=X_train.shape[1], seq_len=X_train.shape[2],
                     channels=channels, num_res_blocks=num_res_blocks,
                     kernel_size=kernel_size, mlp_hidden=mlp_hidden).to(device)

model.compile(optimizer_class=Adam,
              criterion_class=MSELoss,
              lr=lr)

history = model.fit(X_train, y_train, x_val=X_val, y_val=y_val,
                    num_epochs=num_epochs, batch_size=batch_size, patience=patience)

In [ ]:
plt.plot(history['train_loss'])
plt.plot(history['val_loss'])
plt.legend(['train', 'val'])
plt.title(f"Best val loss = {min(history['val_loss']):.2f}")
plt.show()

Попробуйте добавить `BatchNorm` и `LayerNorm` в `ResConvBlock`. Уменьшают ли они $MSE$ на валидации?

In [ ]:
# your code here

## Задача 3. RNN

Дополните заготовки `ResGRUBlock` (он должен иметь двунаправленный GRU и корректное остаточное соединение) и `GRURegressor` (он должен иметь вариабельное число `ResGRU`-блоков). `STEMBlock` и `MLPBlock` оставьте неизменными. Обучите сеть, используя те же гиперпараметры, что у CNN выше.

In [ ]:
class ResGRUBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.channels = channels
        self.gru = # your code here

    def forward(self, x):
        out, _ = self.gru(x)
        out = out[..., :self.channels] + out[..., self.channels:]
        return out + x


class GRURegressor(TrainableNN):
    def __init__(self, input_dim, seq_len,
                 channels=128,
                 num_res_blocks=7, mlp_hidden=128):
        super().__init__()
        self.stem = STEMBlock(input_dim, channels)

        self.res = # your code here

        self.pool = nn.MaxPool1d(seq_len)
        self.mlp = MLPBlock(channels, mlp_hidden)

    def forward(self, x):
        x = self.stem(x)
        x = x.transpose(1, 2)
        x = self.res(x)
        x = x.transpose(1, 2)
        x = self.pool(x).squeeze(-1)
        return self.mlp(x)

In [ ]:
# your code here

Попробуйте добавить `BatchNorm` и `LayerNorm` в `ResGRUBlock`. Какой вариант лучше?

In [ ]:
# your code here

## Задача 3.5. RandomSearch (Опционально. Если у вас сохранилось достаточно GPU-часов)

Дополните код, чтобы осуществить случайный поиск гиперпараметров по приведенной сетке. Осуществите 5-15 итераций. Удалось ли уменьшить $MSE$ на валидации?

In [ ]:
search_space = {
    'lr': [3e-5, 1e-4, 3e-4, 1e-3],
    'channels': [32, 64, 128],
    'num_res_blocks': [1, 2, 3],
    'mlp_hidden': [64, 128, 256],
    'batch_size': [64, 128, 256],
}

patience = 10
num_epochs = 100
best_loss = float('inf')
best_params = None
n_trials = 10

for trial in trange(n_trials):
    params = {k: random.choice(v) for k, v in search_space.items()}

    model = GRURegressor(
        input_dim=X_train.shape[1],
        seq_len=X_train.shape[2],
        channels=params['channels'],
        num_res_blocks=params['num_res_blocks'],
        mlp_hidden=params['mlp_hidden'],
    ).to(device)

    model.compile(optimizer_class=Adam, criterion_class=MSELoss, lr=params['lr'])

    history = model.fit(X_train, y_train,
                        x_val=X_val, y_val=y_val,
                        batch_size=params['batch_size'],
                        num_epochs=num_epochs,
                        patience=patience)

    # your code here

## Задача 4. Сравнение

Совершите предсказание на тестовой выборке лучшими вариантами `CNNRegressor` и `GRURegressor`. Оцените среднее время, которое модели тратят на предсказание. Рассчитайте метрику $R^2$ (берите максимальное предсказание для пары комплементарных последовательностей). Изобразите на графике зависимость качества от времени вычислений.

In [ ]:
# your code here